In [1]:
"""
Inference for Phi-3.5 + LoRA agent adapter.

Load the tokenizer from the adapter folder so tool token IDs match training.
"""
import logging
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
hf_logging.set_verbosity_error()

# Match training: slm-agent/config/training_config.yaml
BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"
ADAPTER_PATH = "./results"  # final save dir, or e.g. ./results/checkpoint-1000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float16


TOOL_SPECIAL_TOKENS = [
    "<tool_use>",
    "</tool_use>",
    "<tool_name>",
    "</tool_name>",
    "<parameters>",
    "</parameters>",
]


def load_tokenizer(adapter_path: str, base_model: str) -> AutoTokenizer:
    p = Path(adapter_path)
    if (p / "tokenizer_config.json").is_file():
        tok = AutoTokenizer.from_pretrained(str(p), trust_remote_code=True)
        logging.info("Tokenizer loaded from adapter: %s (vocab %s)", p.resolve(), len(tok))
        return tok
    logging.warning("No tokenizer in %s — falling back to base + add tokens (IDs may not match old runs).", p)
    tok = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tok.add_special_tokens({"additional_special_tokens": TOOL_SPECIAL_TOKENS})
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model(adapter_path: str = ADAPTER_PATH, merge_lora: bool = False):
    tokenizer = load_tokenizer(adapter_path, BASE_MODEL)
    for t in TOOL_SPECIAL_TOKENS:
        logging.info("%s -> %s", t, tokenizer.convert_tokens_to_ids(t))

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
        trust_remote_code=True,
        attn_implementation="eager",
    ).to(DEVICE)

    if len(tokenizer) != base_model.get_input_embeddings().weight.shape[0]:
        base_model.resize_token_embeddings(len(tokenizer))

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    if merge_lora:
        model = model.merge_and_unload()
        logging.info("LoRA merged into base weights.")

    logging.info("Model ready.")
    return model, tokenizer


# =========================
# PROMPT
# =========================
def build_prompt(user_input: str):
    return f"""### Instruction:
{user_input}

### Response:
"""


# =========================
# GENERATION (SAFE)
# =========================
def generate(model, tokenizer, text):
    prompt = build_prompt(text)

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            use_cache=False,  # 🔥 CRITICAL FIX
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    if "### Response:" in decoded:
        decoded = decoded.split("### Response:")[1]

    return decoded.strip()


def run_tests():
    model, tokenizer = load_model(ADAPTER_PATH, merge_lora=False)

    test_queries = [
        "Help me with: summarize latest AI trends",
        "Help me with: read a configuration file and summarize it",
        "Help me with: compare cost of living in Bangalore and Tokyo",
    ]

    for q in test_queries:
        print(f"\nUser: {q}")
        try:
            response = generate(model, tokenizer, q, greedy=True)
            print(f"Agent:\n{response}")
        except Exception as e:
            print(f"Error: {e}")


def chat():
    model, tokenizer = load_model(ADAPTER_PATH)
    print("Chat (exit / quit to stop)\n")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ("exit", "quit"):
            break
        try:
            reply = generate(model, tokenizer, user_input, greedy=False)
            print(f"Agent:\n{reply}\n{'-' * 50}")
        except Exception as e:
            print(f"Error: {e}")


if __name__ == "__main__":
    run_tests()

2026-03-22 14:18:13,209 - INFO - NumExpr defaulting to 12 threads.

✅ Applied patch 1: added shard_checkpoint
🔄 Loading tokenizer...

2026-03-22 14:18:15,635 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:15,635 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/config.json "HTTP/1.1 200 OK"
2026-03-22 14:18:15,880 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/configuration_phi3.py "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:15,890 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/configuration_phi3.py "HTTP/1.1 200 OK"
2026-03-22 14:18:16,155 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:16,164 - INFO - HTTP Request: HEAD https://huggingface.co/a

🔄 Loading base model (no quantization for stability)...

2026-03-22 14:18:17,654 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:17,658 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/config.json "HTTP/1.1 200 OK"
2026-03-22 14:18:17,896 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/configuration_phi3.py "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:17,905 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/configuration_phi3.py "HTTP/1.1 200 OK"
2026-03-22 14:18:18,144 - INFO - HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/modeling_phi3.py "HTTP/1.1 307 Temporary Redirect"
2026-03-22 14:18:18,154 - INFO - HTTP Request: HEAD https://huggingface

...

32011
🔄 Loading LoRA adapter...

C:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\awq\modules\linear\exllama.py:12: UserWarning: AutoAWQ could not load ExLlama kernels extension. Details: DLL load failed while importing exl_ext: The specified procedure could not be found.
  warnings.warn(f"AutoAWQ could not load ExLlama kernels extension. Details: {ex}")
C:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\awq\modules\linear\exllamav2.py:13: UserWarning: AutoAWQ could not load ExLlamaV2 kernels extension. Details: DLL load failed while importing exlv2_ext: The specified procedure could not be found.
  warnings.warn(f"AutoAWQ could not load ExLlamaV2 kernels extension. Details: {ex}")
C:\Users\SARANSH\Anaconda3\envs\llm_env\Lib\site-packages\awq\modules\linear\gemm.py:14: UserWarning: AutoAWQ could not load GEMM kernels extension. Details: DLL load failed while importing awq_ext: The specified procedure could not be found.
  warnings.warn(f"AutoAWQ could not load GEMM kernels extension. Details: {ex}")
C:

✅ Model loaded successfully!


👤 Search latest AI trends

2026-03-22 14:18:45,445 - WARNING - You are not running the flash-attention implementation, expect numerical differences.

🤖 Thought: I need to use web_search to gather information about the latest AI trends.

Action:
| 
O 
5 �� �� 
<parameters> 
<|placeholder1|> 
K 
o �� 
o 
2 
o 
I 
t 
K 
y 
p 
y 
H 
m 
<parameters> 
m 
<tool_name> 
t �� 
f 
+ 
> 
T 
<tool_use> 
<|placeholder1|> 
; 
~ 
\ 
<tool_use> 
L �� �� 
< 
 Хронологија 
l

👤 Read configuration file and summarize it
🤖 Thought: Step 1: I should use web_search to gather information about reading configuration file and summarizing it.
Action: Progressive IoT Device, Progressive IoT Device
Wrapper Reason: To gather information about reading configuration file and summarizing it using progressive IoT device.
Observation: Searched for insights based on progressive IoT device regarding reading configuration file and summarizing it.

Thought: Step 2: I have completed the multi-step reasoning process for reading configuration file and summarizing it using progressive IoT device.
Final Answer: Based on the progressive IoT device insights, I have completed the

In [ ]:
# Run after the setup cell (Jupyter does not execute `if __name__ == "__main__"`).
run_tests()